# Hello MultiSwitch - many adapters in one request, and across turns

**Duration:** ~10 min for section 1 (CPU, no download); ~30 min more for sections 2-7, which compose a checkpoint

Every other notebook in this repo activates **one** adapter per request. This one puts **several control tokens in a single prompt**, shows the model routing each span to a different adapter, and then drives a **multi-turn conversation** where every turn names its own adapter.

*Why section 1 comes first:* the routing engines depend only on control-token ids and a small codebook - not on adapter weights or the base model. So you can compare both engines and decide which one you need in about a second on CPU, before spending half an hour composing. Stop after section 1 if the decision is all you came for.

*Why compose instead of downloading:* the pre-composed `ibm-granite/granite-switch-4.1-3b-preview` ships with `switch_type="single"`, which cannot do this. The `multi` engine is selected at **compose time** by one flag, so you have to build the checkpoint yourself.

**What you'll learn:**
- The exact sequence shape the default `single` engine gets wrong, measured over an exhaustive sweep rather than one cherry-picked example
- How to compose a checkpoint whose routing supports arbitrarily many transitions per request (`--switch-type multi`)
- How a control token *writes* an adapter id and every later token *reads back* the most recent write
- Why the generated chat template cannot express a two-adapter prompt, and how to place control tokens by hand
- How to read the model's actual per-token routing decision out of `model.model._last_adapter_indices`
- How to run one turn after another with `Conversation`, and what the default KV policy does to earlier turns

**Adapters used:** section 1 loads none - control tokens are plain integer ids. Sections 2-7 pin two, the LoRA flavors of `guardian-core` from [Guardian](https://huggingface.co/ibm-granite/granitelib-guardian-r1.0) and `requirement-check` from [Core](https://huggingface.co/ibm-granite/granitelib-core-r1.0). What is under test is the *routing* rather than either adapter's task quality; they are pinned by name so the prompts and the adapters can be described in the same sentence, which indexing into discovery order did not allow.

## Prerequisites

1. **Install** the compose and HuggingFace extras:


In [ ]:
%pip install "granite-switch[hf,compose]"

2. **Hugging Face login** - the base model and adapter libraries are on the Hub:

In [ ]:
from huggingface_hub import notebook_login

notebook_login()

3. **Disk and bandwidth - sections 2-7 only.** Section 1 downloads nothing. Composing downloads the base model (~6 GB for `granite-4.1-3b` bf16) plus two adapter libraries and writes a new checkpoint. Plan for ~20 GB free. Re-runs hit the HF cache.
4. **GPU is optional.** Section 6 runs a single forward pass and section 7 generates a few tokens per turn; both are fine on CPU, just slower. A GPU makes the load, the forward and the generation faster.

New to Granite Switch? Start with [`hello_adapter.ipynb`](./hello_adapter.ipynb) for the one-adapter case, then come back. Full environment details are in [`../PREREQUISITES.md`](../PREREQUISITES.md).


---

## How multi-adapter routing works

A control token **writes** an adapter id at its position. Every token **reads back** the most recent write at or before it. Nothing else in the sequence changes meaning:

```
position:        0        1          2       3           4       5
token:         "Rate"  <|core|>   "this"  <|guard|>   "reply"  "."
writes:          -        1          -        2           -      -
reads  :         0        1          1        2           2      2
                 ^                                       ^
                 nothing written yet -> base      latest write wins
```

That is the whole contract. `0` means base weights (no adapter); `1..N` index the embedded adapters in `config.adapter_names` order.

The `single` engine can only express the special case where those writes are non-decreasing. Section 1 measures exactly where it breaks.


## 1 · Which engine do you need? (no download)

Granite Switch ships two adapter-selection engines, chosen at compose time by `--switch-type`:

| `--switch-type` | Engine | Mechanism | Transitions per request |
|---|---|---|---|
| `single` (default) | `SingleSwitch` | One attention head with a `+/-gain` cumulative signal | One - routing is **sticky** |
| `multi` | `MultiSwitch` | Counting head recovers `n`, then a Kerdock/DG codebook read | Arbitrarily many - **latest write wins** |

Both read the same control tokens and emit the same thing: one integer adapter index per token. They differ only in what happens when a sequence holds **more than one** control token:

```
sequence:   <|citations|>  "text"  <|query_rewrite|>  "text"
                writes 3             writes 1

wanted:          3           3            1             1     latest write wins
multi :          3           3            1             1     exact
single:          3           3            2             2     averaged 3 and 1 -> 2
                                          ^^^
                              routed to answerability, which nobody asked for
```

That averaged `2` is the whole story: `single` blends competing control values rather than selecting the most recent one.

`create_switch` is the factory the model itself uses - it reads `config.switch_type` and returns the matching engine. Flipping that one string is the entire difference between the two objects built below. Everything in this section is prefixed `DEMO_`/`demo_` so it cannot collide with the real checkpoint's names later.


In [ ]:
import itertools

import torch

from granite_switch.config import GraniteSwitchConfig
from granite_switch.hf.switch import create_switch

DEMO_CTRL_IDS = [201, 202, 203, 204]  # stand-ins for <|query_rewrite|> etc.
DEMO_ADAPTERS = ["query_rewrite", "answerability", "citations", "guardian-core"]
DEMO_TEXT = 50  # any non-control token id
DEMO_INDEX_TO_NAME = {0: "base", **{k + 1: n for k, n in enumerate(DEMO_ADAPTERS)}}


def build_demo_switch(switch_type: str):
    # A real GraniteSwitchConfig with tiny backbone geometry - no adapter weights,
    # no base model, no GPU. The engine only needs the control-token ids and the
    # coded params, so this runs in about a second on CPU.
    config = GraniteSwitchConfig(
        num_adapters=len(DEMO_ADAPTERS),
        switch_type=switch_type,
        adapter_token_ids=DEMO_CTRL_IDS,
        adapter_substitute_token_ids=[1, 2, 3, 4],
        adapter_names=DEMO_ADAPTERS,
        adapter_ranks=[8] * len(DEMO_ADAPTERS),
        vocab_size=2000,
        hidden_size=256,
        num_attention_heads=4,
        num_key_value_heads=2,
        num_hidden_layers=2,
    )
    # SingleSwitch dispatches through ALL_ATTENTION_FUNCTIONS[config._attn_implementation];
    # from_pretrained normally sets this, so a hand-built config must set it too.
    config._attn_implementation = "sdpa"
    return create_switch(config, layer_idx=0)


def demo_route(switch, seq: list[int]) -> list[int]:
    # forward returns (adapter_indices, modified_input_ids); only routing is needed.
    indices, _modified_ids = switch.forward(
        input_ids=torch.tensor([seq]),
        adapter_token_ids=torch.tensor(DEMO_CTRL_IDS),
    )
    return indices[0].tolist()


def demo_latest_wins(seq: list[int]) -> list[int]:
    # Ground truth: every token routes to the expert written by the most recent
    # control token at or before it.
    out, current = [], 0
    for token_id in seq:
        if token_id in DEMO_CTRL_IDS:
            current = DEMO_CTRL_IDS.index(token_id) + 1
        out.append(current)
    return out


demo_single = build_demo_switch("single")
demo_multi = build_demo_switch("multi")

print(
    f"single -> {type(demo_single).__name__:<18} cache slots: {demo_single.num_cache_layers}"
)
print(
    f"multi  -> {type(demo_multi).__name__:<18} cache slots: {demo_multi.num_cache_layers}"
)

Rather than trusting one example, enumerate **every** three-hop sequence over four adapters - all 4^3 = 64 orderings - and count mis-routes against latest-wins ground truth.


In [ ]:
def three_hop(hops: tuple[int, ...]) -> list[int]:
    # Text, then one control token per hop with text after it.
    seq = [DEMO_TEXT]
    for adapter_index in hops:
        seq += [DEMO_CTRL_IDS[adapter_index - 1], DEMO_TEXT]
    return seq


wrong = {"single": [], "multi": []}
all_hops = list(itertools.product(range(1, len(DEMO_ADAPTERS) + 1), repeat=3))

for hops in all_hops:
    seq = three_hop(hops)
    wanted = demo_latest_wins(seq)
    for name, switch in (("single", demo_single), ("multi", demo_multi)):
        if demo_route(switch, seq) != wanted:
            wrong[name].append(hops)

total = len(all_hops)
print(f"three-hop sequences tested: {total}\n")
for name in ("single", "multi"):
    count = len(wrong[name])
    print(f"  {name:<7} mis-routed {count:>2}/{total}  ({100 * count / total:.0f}%)")

# The prose quotes these two numbers, so pin them rather than leaving a reader to
# notice the printout drifted. Nothing here is random: fixed config, fixed
# sequences, so both are reproducible exactly.
assert wrong["multi"] == [], (
    f"multi mis-routed {len(wrong['multi'])} of {total} sequences; latest-wins is "
    "supposed to be exact for any number of transitions, so this is the engine's "
    "core contract failing, not a tuning issue"
)
assert len(wrong["single"]) == 54, (
    f"single mis-routed {len(wrong['single'])} of {total}, expected 54 - the "
    "prose below quotes that figure"
)

print(f"\n{'hops':<12} {'wanted':<26} {'single':<26} multi")
for hops in wrong["single"][:6]:
    seq = three_hop(hops)
    print(
        f"{hops!s:<12} {demo_latest_wins(seq)!s:<26} "
        f"{demo_route(demo_single, seq)!s:<26} {demo_route(demo_multi, seq)}"
    )

54 of 64 for `single`, 0 for `multi`. The ten it gets right are the *near-constant* orderings - runs of one adapter, plus hops between neighbouring indices where the average happens to coincide with the latest write. Strictly increasing hops like `(1, 2, 3)` are **not** among them.

That counts wrong *positions*, which is the right measurement and the wrong unit: nobody debugs a position. An agent appends to **one** growing sequence and asks one question per step - *did this step run on the adapter I named?*


In [ ]:
def build_demo_transcript(steps):
    # steps: (adapter_name, token_count). Returns token ids plus the
    # (start_position, adapter_name) of each step so results can be attributed.
    ids: list[int] = []
    spans: list[tuple[int, str]] = []
    for adapter_name, token_count in steps:
        spans.append((len(ids), adapter_name))
        ids.append(DEMO_CTRL_IDS[DEMO_ADAPTERS.index(adapter_name)])
        ids.extend([DEMO_TEXT] * token_count)
    return ids, spans


STEPS = [
    ("query_rewrite", 3),  # step 1: clean up the user's question
    ("citations", 3),  # step 2: attribute the drafted answer
    ("guardian-core", 2),  # step 3: safety screen before returning it
]

transcript, step_spans = build_demo_transcript(STEPS)
got_multi = demo_route(demo_multi, transcript)
got_single = demo_route(demo_single, transcript)

print(f"{'step':<6} {'asked for':<16} {'multi got':<16} {'single got':<16}")
for step_number, (start, asked) in enumerate(step_spans, start=1):
    print(
        f"{step_number:<6} {asked:<16} "
        f"{DEMO_INDEX_TO_NAME[got_multi[start]]:<16} {DEMO_INDEX_TO_NAME[got_single[start]]:<16}"
    )

final_start, final_asked = step_spans[-1]
delivered = DEMO_INDEX_TO_NAME[got_single[final_start]]
print(f"\nfinal step wanted {final_asked!r}; single delivered {delivered!r}")

assert got_multi == demo_latest_wins(transcript), (
    "multi must be exact on a per-step transcript"
)
assert delivered != final_asked, (
    "single delivered the final step to the adapter that was asked for, so this "
    "cell no longer demonstrates the failure it exists to show - the transcript "
    "shape must have drifted into one of the orderings single happens to get right"
)

Read the last row first. The **final** step is where a caller reads the answer, so a mis-route there does the most damage - and nothing raises. `single` returned `citations` output to a request for `guardian-core`: a safety screen answered by a citation adapter, with the right shape and the wrong weights. `single` is exact at one step and wrong from two onward, which is what makes this easy to ship broken - an agent that starts with one tool call per request passes every test, then mis-routes the moment a second step joins the same sequence.

### Choosing an engine

`multi` is not a free upgrade - it owns two KV cache slots instead of one and forces fp32 attention on both of its heads. Pick by whether a single sequence ever holds more than one control token.

| Your workload | Engine | Why |
|---|---|---|
| One adapter per request (the mellea intrinsics path, [`hello_mellea.ipynb`](./hello_mellea.ipynb)) | `single` | Only one control token per sequence, so there is nothing to mis-route. Cheaper: one cache slot. |
| Standard multi-turn chat where the template emits a token for the current turn only | `single` | Prior turns carry no control token and route to base already. |
| One sequence deliberately holding several control tokens - per-step agent routing | `multi` | Latest-wins is exact for any number of transitions. |
| Spans of one prompt attributed to different adapters | `multi` | Same reason: `single` averages the competing writes. |
| A multi-turn dialogue that keeps earlier turns' control tokens | `multi` | Each turn's region must route to its own adapter, non-contiguously. See section 7. |

Two limits to design around, both from the engine's own documentation:

- **Returning to base mid-sequence is opt-in at compose time.** By default `add_control_tokens` emits exactly one control token per adapter, so `len(adapter_token_ids) == num_adapters`, there is no base-reset token, and once a step selects an adapter later steps can only select a *different* adapter - not plain base weights. Composing with `--base-reset-token` prepends `<|base_reset|>`, giving the `num_adapters + 1` layout the engine reads as offset 0 (`multi.py:149-176`); you then place that token yourself, since no chat template emits it.
- **Adapters route spans, not generations.** Each step's control token changes which weights process the tokens *after* it in the same forward pass. Getting a separate generation per step is still a separate request - a pipeline whose step 2 consumes step 1's output text stays sequential. [`rag_flow.ipynb`](./rag_flow.ipynb) is that data-dependent shape, and it is correct as separate calls.

Everything from here on needs a real checkpoint.


## 2 · Imports and configuration

`MODEL_OUT` is where the composed checkpoint lands. Two flags keep the build deterministic. `--technology-filter lora` fixes where the control token goes - LoRA puts it at a fixed position, so the routing trace in section 6 is easy to read. `--include-adapters` fixes *which* adapters are embedded, so `ADAPTER_A`/`ADAPTER_B` are the two named below rather than whichever two the composer discovered first. [`compose_granite_switch.ipynb`](./compose_granite_switch.ipynb) section 4 covers the unfiltered build, where LoRA and aLoRA adapters coexist and place their tokens differently.


In [ ]:
from transformers import AutoTokenizer

from granite_switch import Conversation, KVHistoryPolicy
from granite_switch.hf import GraniteSwitchForCausalLM
from granite_switch.hf.switch import MultiSwitch

BASE_MODEL = "ibm-granite/granite-4.1-3b"
GUARDIAN_LIB = "ibm-granite/granitelib-guardian-r1.0"
CORE_LIB = "ibm-granite/granitelib-core-r1.0"
MODEL_OUT = "./granite-switch-multi"

# Pinned rather than taken as adapter_names[0], [1]. Discovery order is a property
# of the libraries, so indexing into it silently picks whatever the composer found
# first -- which put two factuality adapters here and left section 7 asking a
# factuality corrector to summarise a document. Naming them keeps this notebook's
# prompts and its adapters describable in the same sentence.
ADAPTER_A = "guardian-core"  # safety / risk screen
ADAPTER_B = "requirement-check"  # does a response satisfy stated requirements

print(f"base:   {BASE_MODEL}")
print(f"output: {MODEL_OUT}")
print(f"adapters: A = {ADAPTER_A} (screen), B = {ADAPTER_B} (spec check)")

## 3 · Compose with the `multi` engine

One flag distinguishes this build from every other notebook's: `--switch-type multi`. It selects the Kerdock/DG coded-memory engine and persists to `config.json`, so `from_pretrained` later rebuilds the matching engine automatically.


In [ ]:
!python -m granite_switch.composer.compose_granite_switch \
  --base-model {BASE_MODEL} \
  --adapters {GUARDIAN_LIB} {CORE_LIB} \
  --include-adapters {ADAPTER_A} {ADAPTER_B} \
  --technology-filter lora \
  --switch-type multi \
  --output {MODEL_OUT}

## 4 · Confirm the engine persisted

Worth asserting rather than assuming: if `--switch-type` had not been written to `config.json`, `from_pretrained` would silently build a `SingleSwitch` and the rest of this notebook would produce quietly wrong routing instead of an error.


In [ ]:
config = GraniteSwitchConfig.from_pretrained(MODEL_OUT)
assert config.switch_type == "multi", (
    f"config.switch_type is {config.switch_type!r} - the --switch-type flag did not persist"
)

model = GraniteSwitchForCausalLM.from_pretrained(MODEL_OUT).eval()
assert isinstance(model.model.switch, MultiSwitch), (
    f"from_pretrained built {type(model.model.switch).__name__}, expected MultiSwitch"
)

print(f"switch engine     : {type(model.model.switch).__name__}")
print(f"embedded adapters : {config.num_adapters}")
print(f"cache slots       : {model.model.switch.num_cache_layers}  (counting + memory)")

## 5 · The control tokens

Composing added one special token per adapter, named `<|adapter_name|>`. `config.adapter_token_ids[k]` is the token that fires adapter `k+1` - the off-by-one is the base slot at index 0.


In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_OUT)

print(f"{'adapter':<28} {'control token':<30} {'id':>6}  fires")
for k, (name, token_id) in enumerate(
    zip(config.adapter_names, config.adapter_token_ids)
):
    token = f"<|{name}|>"
    # The composed tokenizer must round-trip the token as a single special id.
    assert tokenizer.convert_tokens_to_ids(token) == token_id, (
        f"{token} did not round-trip"
    )
    print(f"{name:<28} {token:<30} {token_id:>6}  adapter {k + 1}")

# --include-adapters is a glob filter, so a typo or a renamed adapter would compose
# a checkpoint that silently lacks one of them. Fail here instead.
missing = [n for n in (ADAPTER_A, ADAPTER_B) if n not in config.adapter_names]
assert not missing, (
    f"{missing} not embedded; --include-adapters matched {list(config.adapter_names)}"
)
print(f"\nusing A = {ADAPTER_A!r}, B = {ADAPTER_B!r}")

## 6 · One prompt, two adapters

The chat template binds exactly **one** adapter per call. `adapter_map` is a dict, so passing a list is not merely unsupported, it raises:

```python
tokenizer.apply_chat_template(messages, adapter_name=["citations", "guardian-core"])
# TypeError: unhashable type: 'list'
```

The fix is not to abandon the template - it is what gets the role markers, the turn terminator, and the generation prompt right. Template **first** with the adapter for the turn, then **splice** the second control token in immediately before the generation prompt. Routing then switches exactly at the boundary between the conversation and the model's answer.


In [ ]:
ROLE_OPEN_MARKERS = ("<|start_of_role|>", "<|im_start|>")


def role_open_id() -> int:
    # Granite ships two template families - 4.0/4.1 role markers and 4.2 ChatML -
    # so detect which one this checkpoint uses instead of hardcoding a marker.
    for marker in ROLE_OPEN_MARKERS:
        token_id = tokenizer.convert_tokens_to_ids(marker)
        if token_id is not None and token_id != tokenizer.unk_token_id:
            return token_id
    raise RuntimeError(f"no role-open marker found among {ROLE_OPEN_MARKERS}")


def templated_ids(messages: list[dict], adapter_name: str | None = None) -> list[int]:
    kwargs = {"adapter_name": adapter_name} if adapter_name else {}
    out = tokenizer.apply_chat_template(
        messages, add_generation_prompt=True, tokenize=True, **kwargs
    )
    # transformers 5.x returns a BatchEncoding here; 4.x returns a plain list.
    return list(out["input_ids"]) if hasattr(out, "keys") else list(out)


def build_multi_adapter_prompt(
    messages: list[dict], turn_adapter: str, generation_adapter: str
) -> list[int]:
    ids = templated_ids(messages, turn_adapter)
    # The LAST role-open marker begins the generation prompt; splicing before it
    # means the answer is generated under generation_adapter while the
    # conversation above stays on turn_adapter.
    cut = max(i for i, token_id in enumerate(ids) if token_id == role_open_id())
    control = tokenizer.convert_tokens_to_ids(f"<|{generation_adapter}|>")
    return ids[:cut] + [control] + ids[cut:]


# Same draft section 7 works through, so the two sections read as one story.
MESSAGES = [
    {
        "role": "user",
        "content": (
            "Draft reply: 'Your refund request has been denied.' "
            "Does it meet our policy, and is it harmful?"
        ),
    }
]

one_adapter = templated_ids(MESSAGES, ADAPTER_A)
input_ids = build_multi_adapter_prompt(MESSAGES, ADAPTER_A, ADAPTER_B)

print(f"templated (1 adapter) : {tokenizer.decode(one_adapter)!r}")
print(f"spliced   (2 adapters): {tokenizer.decode(input_ids)!r}")
assert len(input_ids) - len(one_adapter) == 1, "splice should add exactly one token"

Now run it and read the switch's decision per position.

In [ ]:
with torch.no_grad():
    model(input_ids=torch.tensor([input_ids]))

# The model exposes the switch's decision for the last forward pass.
routing = model.model._last_adapter_indices[0].tolist()

index_to_name = {0: "base (no adapter)"}
index_to_name.update({k + 1: n for k, n in enumerate(config.adapter_names)})

print(f"{'pos':>4}  {'token':<26} {'idx':>4}  adapter")
for position, (token_id, adapter_index) in enumerate(zip(input_ids, routing)):
    token = tokenizer.convert_ids_to_tokens(token_id)
    mark = "  <- writes" if token_id in config.adapter_token_ids else ""
    print(
        f"{position:>4}  {token:<26} {adapter_index:>4}  {index_to_name[adapter_index]}{mark}"
    )

Two spans, two sets of weights: the user turn runs on the first adapter, and everything from the spliced token onward - including the tokens the model is about to generate - runs on the second.

Note there is **no base span** here. This checkpoint was composed with `--technology-filter lora`, and LoRA places its control token at position 0, so an adapter is active from the very first token. That is the placement difference [`compose_granite_switch.ipynb`](./compose_granite_switch.ipynb) section 4 is about.

The assertion below states the contract rather than leaving it to eyeballing - every position routes to the most recent control token at or before it.


In [ ]:
expected, current = [], 0
for token_id in input_ids:
    if token_id in config.adapter_token_ids:
        current = config.adapter_token_ids.index(token_id) + 1
    expected.append(current)

assert routing == expected, f"routing {routing} != latest-wins {expected}"

spans = {index_to_name[i]: routing.count(i) for i in sorted(set(routing))}
print(f"latest-wins routing confirmed over {len(input_ids)} positions")
print(f"spans: {spans}")

## 7 · One turn after another

Sections 5 and 6 were one request. A real caller runs a dialogue: ask, generate, append, ask again - and each turn may want a different adapter. `Conversation` is the piece that does the bookkeeping. It takes the composed tokenizer, and the loop is four calls per turn:

```
conversation.user(question)                        # append the user turn
prompt = conversation.build_prompt(adapter=name)   # token ids to send
answer = <generate from prompt>                    # HF generate, or POST to vLLM
conversation.record_answer(answer_ids, adapter=name)   # commit the turn
```

`record_answer` is what commits a turn - a prompt whose answer is never recorded (a guardian screen you discarded) leaves the transcript untouched. Prefer handing it the **ids** the model emitted over the decoded text: `encode(decode(ids))` does not always reproduce `ids`.

One gotcha worth naming: `model.model._last_adapter_indices` holds the routing of the **last** forward pass, and `generate` runs one per decoded token. So the trace below takes a separate forward over the prompt before generating - redundant compute, but the only way to see the prompt's routing rather than the final decode step's single row.


In [ ]:
MAX_NEW_TOKENS = 16  # small on purpose: CPU generation is the slow part here

# Each turn names the adapter whose job it is: B checks a draft against a stated
# requirement, A screens the draft for harm. The point is the routing, but a turn
# an adapter can plausibly answer keeps the output readable.
SPEC = (
    "Our support policy says every refund denial must cite the policy section it "
    "relies on. Draft reply: 'Your refund request has been denied.' "
)
TURNS = [
    (SPEC + "Does the draft meet the policy?", ADAPTER_B),
    ("Is that draft reply harmful or abusive toward the customer?", ADAPTER_A),
    ("We revised it to cite section 4.2. Re-check it against the policy.", ADAPTER_B),
]


def routing_spans(indices: list[int]) -> list[tuple[int, int, int]]:
    """Collapse per-position indices into (start, end_inclusive, adapter) runs."""
    runs: list[list[int]] = []
    for position, index in enumerate(indices):
        if runs and runs[-1][2] == index:
            runs[-1][1] = position
        else:
            runs.append([position, position, index])
    return [tuple(run) for run in runs]


# No policy argument: RE_PREFILL is the default, and it is what a plain
# apply_chat_template loop already does. Section 7's closing note covers the other one.
conversation = Conversation(tokenizer, config=config)

for turn, (question, adapter) in enumerate(TURNS, start=1):
    conversation.user(question)
    prompt = list(conversation.build_prompt(adapter=adapter))

    with torch.no_grad():
        model(input_ids=torch.tensor([prompt]))
    turn_routing = model.model._last_adapter_indices[0].tolist()

    with torch.no_grad():
        generated = model.generate(
            input_ids=torch.tensor([prompt]),
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )
    answer_ids = generated[0, len(prompt) :].tolist()
    conversation.record_answer(answer_ids, adapter=adapter)

    controls = [i for i, t in enumerate(prompt) if t in config.adapter_token_ids]
    print(
        f"turn {turn}  adapter={adapter:<24} {len(prompt):>4} prompt tokens   control tokens at {controls}"
    )
    for start, end, index in routing_spans(turn_routing):
        print(
            f"        {start:>4} - {end:<4} ({end - start + 1:>3} pos)  {index_to_name[index]}"
        )
    print(
        f"        answer: {tokenizer.decode(answer_ids, skip_special_tokens=True)!r}\n"
    )

Three things to read off that output.

**The loop is adapter-agnostic.** Turn 2 named a different adapter than turn 1 and nothing else in the calling code changed. The prompt grows because the transcript grows.

**Every turn carries exactly one control token, and it sits at position 0.** That is `RE_PREFILL`, the default: each turn is re-rendered from `messages`, and `messages` is *text*. A control token is markup that exists only in the token ids a render produced, so turn 1's token is gone by turn 2. Every ordinary chat API behaves that way, and it is usually fine.

**So the whole prompt routes to the current turn's adapter - history included.** This checkpoint is LoRA, which places the control token at position 0, so there is exactly one span per turn. On turn 2 that means turn 1's question and answer are processed by `requirement-check`'s weights on turn 1 and by `guardian-core`'s on turn 2. The routing is not wrong - it is what a re-rendered transcript can express - but nothing records which adapter produced which region.

The other policy, `KVHistoryPolicy.PRESERVE_MIXED_HISTORY`, sends the ids already sent plus the new turn - so each turn's region keeps routing to the adapter that produced it, and stays eligible for the prefix cache. It needs `multi` precisely because that puts several control tokens in one request. It also needs **aLoRA** adapters, so it refuses this LoRA-only checkpoint:


In [ ]:
# A LoRA control token is emitted at sequence position 0 - inside the already-sent
# prefix on every turn after the first - so the policy could never hold for it.
# Refused on turn 1 rather than turn 2, because a conversation does not change
# adapter technology mid-dialogue.
try:
    preserving = Conversation(
        tokenizer, policy=KVHistoryPolicy.PRESERVE_MIXED_HISTORY, config=config
    )
    preserving.user(TURNS[0][0])
    preserving.build_prompt(adapter=ADAPTER_A)
except (ValueError, RuntimeError) as error:
    print(f"refused, as it should be:\n  {str(error).split('. ')[0]}.")
else:
    raise AssertionError(
        "PRESERVE_MIXED_HISTORY accepted a LoRA adapter - the guard that makes this "
        "policy safe is not firing"
    )

## 8 · Next steps

- **Keep every turn on its own adapter.** [`multi_turn_multiswitch.ipynb`](./multi_turn_multiswitch.ipynb) is section 7 done properly: aLoRA adapters, `PRESERVE_MIXED_HISTORY`, per-turn routing that survives the turn boundary, and the prefix-cache reuse that buys.
- **Serve it with vLLM.** [`multiswitch_serving.ipynb`](./multiswitch_serving.ipynb) puts a `multi` checkpoint behind an OpenAI-compatible server, sends a two-adapter prompt over HTTP, and checks that batching does not leak between requests.
- **Compose for a different goal.** [`compose_granite_switch.ipynb`](./compose_granite_switch.ipynb) is the full tour of the composer's selection flags - `--include-adapters`, `--exclude-adapters`, `--technology-filter`, and where `--switch-type` fits in.
- **Serve single-adapter workloads through mellea.** [`hello_mellea.ipynb`](./hello_mellea.ipynb) is the recommended path when one adapter per request is enough - it handles constrained decoding and output parsing for you.
- **Read the mechanism end to end.** [`../../docs/MULTISWITCH_EXPLAINED.html`](../../docs/MULTISWITCH_EXPLAINED.html) covers the engine, both backends, per-request isolation under vLLM, and the limits.
